In [1]:
import os
import pickle
from PIL import Image

# Path to extracted CIFAR-100 folder
dataset_path = "CIFAR100"

# Output folder
output_dir = "Data"
os.makedirs(output_dir, exist_ok=True)


def unpickle(file):
    with open(file, "rb") as fo:
        data = pickle.load(fo, encoding="bytes")
    return data


# Load class names
meta = unpickle(os.path.join(dataset_path, "meta"))
fine_label_names = [name.decode("utf-8") for name in meta[b"fine_label_names"]]


def save_images(data_file, split):
    data = unpickle(os.path.join(dataset_path, data_file))

    images = data[b"data"]
    labels = data[b"fine_labels"]

    # Create split directory (train/test)
    split_dir = os.path.join(output_dir, split)
    os.makedirs(split_dir, exist_ok=True)

    for i, (img, label) in enumerate(zip(images, labels)):
        class_name = fine_label_names[label]

        # Create class directory inside train/test
        class_dir = os.path.join(split_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        # Convert image to (32, 32, 3)
        img = img.reshape(3, 32, 32).transpose(1, 2, 0)
        image = Image.fromarray(img)

        # Save image
        filename = f"{i:05d}.png"
        image.save(os.path.join(class_dir, filename))

    print(f"{split}: Saved {len(images)} images.")


# Save training and testing images
save_images("train", "train")
save_images("test", "test")

print("Done!")


train: Saved 50000 images.
test: Saved 10000 images.
Done!


In [2]:
import os
import random
import shutil

# Paths
test_dir = "Data/test"
val_dir = "Data/val"

# Percentage of images to move from test to validation
val_split = 0.5  # 50%

random.seed(42)  # For reproducibility

os.makedirs(val_dir, exist_ok=True)

for class_name in os.listdir(test_dir):
    class_test_dir = os.path.join(test_dir, class_name)

    if not os.path.isdir(class_test_dir):
        continue

    class_val_dir = os.path.join(val_dir, class_name)
    os.makedirs(class_val_dir, exist_ok=True)

    images = [
        f for f in os.listdir(class_test_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    random.shuffle(images)

    num_val = int(len(images) * val_split)
    val_images = images[:num_val]

    for img in val_images:
        src = os.path.join(class_test_dir, img)
        dst = os.path.join(class_val_dir, img)
        shutil.move(src, dst)

print("Validation split completed!")

Validation split completed!
